# **IBOVESPA Trend Prediction**

## Exploratory Data Analysis (EDA)

### Objetivo

O objetivo desta análise exploratória é compreender o comportamento histórico do índice **IBOVESPA**, identificar padrões nos dados e preparar as variáveis para a construção de um modelo preditivo.

A análise busca:

- Entender a distribuição das variáveis do dataset  
- Identificar possíveis inconsistências ou valores ausentes  
- Explorar relações entre preço de abertura, fechamento, máxima, mínima e volume  
- Avaliar padrões temporais do índice  
- Preparar os dados para a etapa de **modelagem preditiva**, cujo objetivo é prever se o **fechamento do IBOVESPA no dia seguinte será maior ou menor que o do dia atual**.

## **1. Import Lab**

In [16]:
# Manipulação e cálculos
import pandas as pd #Ler, limpar, transformar tabelas
import numpy as np #Cálculos numéricos rápidos (base para Pandas)

# Visualização
import matplotlib.pyplot as plt #Criação de gráficos personalizados
import matplotlib.ticker as mtick #formatar, posicionar e controlar os rótulos (ticks) dos eixos X e Y dos gráficos.

# Utilidades
import unicodedata
from pathlib import Path

## **2. Personalizar as configurações de exibição do Pandas**

In [17]:
pd.set_option('display.max_columns', None) #Exibir todas as colunas
pd.set_option('display.max_rows', None) #Exibir todas as linhas
pd.set_option('display.float_format', lambda x: f'{x:,.2f}') #2 casas decimais e milhar

## **Base de Dados**
**Dados:** https://br.investing.com/indices/bovespa-historical-data<br>
**Filtro:** 01/01/2024 a 04/03/2026

## **3. Carregar os Arquivo .csv**

In [18]:
RAW = Path("../data/raw")

df_dados_ibovespa = pd.read_csv(
    RAW / "ibov_dados_historicos_012024_032026.csv", 
    sep=","
)

## **4. Visualização e Informações do Dataset**

In [21]:
# Primeiras linhas
df_dados_ibovespa.head()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
0,04.03.2026,185.77,183.11,186.31,183.11,"7,38M","1,46%"
1,03.03.2026,183.10,189.28,189.60,180.52,"14,37B","-3,28%"
2,02.03.2026,189.31,188.79,190.11,186.64,"9,09B","0,28%"
3,27.02.2026,188.79,191.00,191.00,188.48,"11,00B","-1,16%"
4,26.02.2026,191.00,191.25,191.98,188.98,"8,80B","-0,13%"


In [22]:
# Últimas linhas
df_dados_ibovespa.tail()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
538,08.01.2024,132.43,132.02,132.50,131.01,"8,50M","0,31%"
539,05.01.2024,132.02,131.22,132.63,130.58,"9,20M","0,61%"
540,04.01.2024,131.23,132.83,132.88,131.02,"8,97M","-1,21%"
541,03.01.2024,132.83,132.70,133.58,132.25,"8,70M","0,10%"
542,02.01.2024,132.70,134.19,134.19,132.09,"8,44M","-1,11%"


In [23]:
# Tamanho do dataframe
df_dados_ibovespa.shape

(543, 7)

In [24]:
# Informações gerais (tipos e nulos)
df_dados_ibovespa.info()

<class 'pandas.DataFrame'>
RangeIndex: 543 entries, 0 to 542
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Data      543 non-null    str    
 1   Último    543 non-null    float64
 2   Abertura  543 non-null    float64
 3   Máxima    543 non-null    float64
 4   Mínima    543 non-null    float64
 5   Vol.      543 non-null    str    
 6   Var%      543 non-null    str    
dtypes: float64(4), str(3)
memory usage: 29.8 KB


In [25]:
# Estatísticas descritivas (numéricas)
df_dados_ibovespa.describe()

,Último,Abertura,Máxima,Mínima
count,543.00,543.00,543.00,543.00
mean,136.64,136.54,137.46,135.75
std,15.46,15.32,15.57,15.17
min,118.53,118.53,119.45,118.22
25%,127.31,127.31,127.97,126.47
50%,131.45,131.45,132.02,130.58
75%,139.32,139.28,140.02,138.51
max,191.49,191.49,192.62,190.42


In [26]:
# Quantidade de nulos por coluna
df_dados_ibovespa.isna().sum()

Data        0
Último      0
Abertura    0
Máxima      0
Mínima      0
Vol.        0
Var%        0
dtype: int64

In [27]:
# Checar duplicados
df_dados_ibovespa.duplicated().sum()

np.int64(0)

## **5. Tratamento da Base**
Preparar o dataset para análise e modelagem:
1. Converter tipos de variáveis (Data, Vol., Var%)
2. Garantir ordem cronológica crescente
3. Definir `Data` como índice (séries temporais)

### **5.1. Converter os tipos de variáveis**
- **Data:** `object` → `datetime`
- **Vol.:** `object` → `float` (tratando sufixos **K**, **M**, **B**)
- **Var%:** `object` → `float` (removendo `%` e ajustando decimal)

### **5.1.1. Data (object → datetime)**

In [32]:
# Data: object → datetime
df_dados_ibovespa["Data"] = pd.to_datetime(
    df_dados_ibovespa["Data"],
    format="%d.%m.%Y",
    errors="coerce"  # transforma inválidos em NaT ao invés de quebrar
)

### **5.1.2 Vol. (object → float)**

In [33]:
# Vol.: object → float
# Trata K (mil), M (milhões), B (bilhões), além de '-', vazio e NaN
def tratar_volume(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    if valor in ("", "-", "—"):
        return np.nan

    valor = valor.replace(",", ".")  # ajusta decimal

    multipliers = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}

    sufixo = valor[-1]
    if sufixo in multipliers:
        return float(valor[:-1]) * multipliers[sufixo]

    # se vier número "puro"
    return float(valor)

df_dados_ibovespa["Vol."] = df_dados_ibovespa["Vol."].apply(tratar_volume)

### **5.1.3 Var% (object → float)**

In [34]:
# Var%: object → float
# Remove % e ajusta vírgula decimal
df_dados_ibovespa["Var%"] = (
    df_dados_ibovespa["Var%"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .replace({"-": np.nan, "—": np.nan, "": np.nan})
    .astype(float)
)

### **5.2. Variável Data: Ordem cronológica crescente**
Como os cálculos consideram a linha anterior como “dia anterior”, os dados precisam estar
do mais antigo para o mais recente para manter a sequência temporal correta.

In [37]:
# Verificar se já está em ordem crescente
df_dados_ibovespa["Data"].is_monotonic_increasing

True

In [36]:
# Ordenar pela Data (crescente) e resetar índice
df_dados_ibovespa = (
    df_dados_ibovespa
    .sort_values("Data", ascending=True)
    .reset_index(drop=True)
)

### **5.3. Adicionar Data como índice**
Definir a coluna `Data` como índice facilita operações de séries temporais
(resample, rolling, etc.).

In [38]:
df_dados_ibovespa = df_dados_ibovespa.set_index("Data")

### **6. Salvando os dados tratados**
Após realizar o tratamento e limpeza da base, salvamos o dataset processado na pasta **`data/processed`**.

Separar os dados em **raw** e **processed** é uma boa prática em projetos de dados, pois:

- **raw:** mantém os dados originais sem alteração  
- **processed:** contém os dados já tratados e prontos para análise ou modelagem  

Isso facilita a reprodutibilidade do projeto e evita modificar os dados originais.

In [42]:
# Caminho da pasta
PROCESSED = Path("../data/processed")

# Salvar dataset tratado
df_dados_ibovespa.to_csv(
    PROCESSED / "ibov_dados_tratados.csv"
)